# Aurora global feasibility spike — Dayli AI

**One question: does a non-debug Aurora model run on the full global 0.25° grid inside a free 16 GB Kaggle GPU?**

Everything downstream depends on the peak-VRAM number this notebook prints.

| Result | What it means |
|---|---|
| Peak < ~15 GB on 1×T4 | Free pipeline works. Build `run_aurora_india.py`. |
| Fits only on 2×T4 (32 GB) | Switch Kaggle accelerator to `T4 x2`. Still free. |
| OOM on 32 GB | **Stop.** Use ECMWF AIFS via Open-Meteo instead. |

### Rules this notebook follows

1. **Global grid only.** Aurora's pretrained models do not accept a regional box —
   maintainer `wesselb`, [microsoft/aurora#178](https://github.com/microsoft/aurora/issues/178):
   they *"will not work out-of-the-box if you give them data of limited spatial extent"*.
   We run global and slice the output later. A Telangana or India box is not a shortcut.
2. **No `AuroraSmallPretrained`.** Upstream: *"Use this model for debugging purposes.
   We do not recommend any other use."* It must never drive health advice.
3. **Real initial conditions or nothing.** If the ECMWF download fails, this notebook
   fails loudly. It does not substitute random noise or a climatological curve — that is
   exactly the bug this whole effort exists to undo.

### Before running

- Settings → Accelerator → **GPU T4 x1** (switch to `T4 x2` only if told to below).
- Settings → Internet → **On** (needed for ECMWF + HuggingFace downloads).


## 1. Install


In [ ]:
%pip install -q microsoft-aurora ecmwf-opendata cfgrib eccodes huggingface_hub
# cfgrib needs the ecCodes binary; this makes the GRIB engine importable on Kaggle.
import eccodes  # noqa: F401
print('installed')


## 2. Environment check


In [ ]:
import torch, platform

print('torch       :', torch.__version__)
print('cuda avail  :', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Settings -> Accelerator -> GPU T4. Aborting.')

n = torch.cuda.device_count()
total = 0
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1024**3
    total += gb
    print(f'gpu {i}       : {p.name}  {gb:.1f} GiB')
print(f'total vram  : {total:.1f} GiB')

# Precision is NOT ours to set. Aurora's forward() does `batch.type(p.dtype)`,
# casting the batch to the parameter dtype. If we half the model, that cast also
# hits metadata.lon, where 359.75 rounds to 360.0 in bf16 (only ~8 mantissa bits)
# and Aurora's own validator then raises "Longitudes must be in the range [0,360)".
# AuroraV1p5 already does fp16 autocast on encoder/backbone/decoder internally,
# so the model stays float32 and we keep the savings.
print('bf16 supported :', torch.cuda.is_bf16_supported())
print('model dtype    : float32 (AuroraV1p5 autocasts to fp16 internally)')


## 3. Download real ECMWF initial conditions

Aurora needs **two** consecutive atmospheric states (t−6 h and t) as history.
We take `step=0` from the current cycle and from the one 6 h before it.

ECMWF open data is free, no API key, CC-BY-4.0.


In [ ]:
import datetime as dt, os
from ecmwf.opendata import Client

# Verified against the live ECMWF open-data index on 2026-11-06: the oper 0p25
# stream publishes t,u,v,q,z on all 13 Aurora levels, plus 2t,10u,10v,msl at the
# surface. Every input Aurora needs is present. No substitution required.
LEVELS = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]
SURF   = ['2t', '10u', '10v', 'msl']
PL     = ['t', 'u', 'v', 'q', 'z']

client = Client(source='ecmwf')
os.makedirs('ic', exist_ok=True)

def latest_cycle(lag_hours=10):
    """Most recent 6-hourly cycle old enough to have been published.

    lag=10h was checked against the server for all 24 hours of the day; both
    this cycle and the t-6 one were always already published.
    """
    t = dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=lag_hours)
    return t.replace(hour=(t.hour // 6) * 6, minute=0, second=0, microsecond=0)

CYCLE = latest_cycle()
PREV  = CYCLE - dt.timedelta(hours=6)
print('analysis time t   :', CYCLE.isoformat())
print('analysis time t-6 :', PREV.isoformat())

def grab(when, levtype, params, target, levelist=None):
    # Skip files already on disk. Kaggle keeps the filesystem across a kernel
    # restart, so a rerun after an error costs nothing instead of ~1 GB.
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f'  {target:28s} {os.path.getsize(target)/1024**2:7.1f} MB (cached)')
        return
    req = dict(date=when.strftime('%Y%m%d'), time=when.hour, type='fc', step=0,
               levtype=levtype, param=params, target=target)
    if levelist:
        req['levelist'] = levelist
    client.retrieve(**req)
    print(f'  {target:28s} {os.path.getsize(target)/1024**2:7.1f} MB')

print('downloading (expect ~1 GB total):')
for tag, when in (('t', CYCLE), ('p', PREV)):
    grab(when, 'sfc', SURF, f'ic/{tag}_sfc.grib2')
    grab(when, 'pl',  PL,   f'ic/{tag}_pl.grib2', LEVELS)
print('done')


## 4. Load GRIB into arrays on Aurora's grid

Aurora expects latitude **descending** (90 → −90) and longitude **ascending** (0 → 359.75),
which is exactly how ECMWF 0.25° open data is published: 721 × 1440.


In [ ]:
import numpy as np, xarray as xr

# Two GRIB gotchas, both hit and verified against real ECMWF data:
#
# 1. A surface file mixes level types: 2t on heightAboveGround=2, 10u/10v on
#    heightAboveGround=10, msl on meanSea. cfgrib refuses to build one dataset
#    from conflicting coordinates and SILENTLY SKIPS the clashing variables --
#    you get a KeyError on u10 rather than a clear message. So we open the file
#    once per level type with an explicit filter.
#
# 2. ECMWF publishes longitude as [-180, 179.75]. Aurora requires [0, 360)
#    strictly increasing (Metadata.__post_init__ raises ValueError otherwise).
#    We roll the longitude axis and every data array together.

def open_grib(path, **filters):
    return xr.open_dataset(
        path, engine='cfgrib',
        backend_kwargs={'indexpath': '', 'filter_by_keys': filters})

def load(tag):
    sfc_path = f'ic/{tag}_sfc.grib2'
    d2  = open_grib(sfc_path, typeOfLevel='heightAboveGround', level=2)
    d10 = open_grib(sfc_path, typeOfLevel='heightAboveGround', level=10)
    dms = open_grib(sfc_path, typeOfLevel='meanSea')

    surf = {'2t':  d2['t2m'].values,  '10u': d10['u10'].values,
            '10v': d10['v10'].values, 'msl': dms['msl'].values}

    pl = open_grib(f'ic/{tag}_pl.grib2', typeOfLevel='isobaricInhPa')
    pl = pl.sel(isobaricInhPa=LEVELS)   # enforce our level order
    atmos = {'t': pl['t'].values, 'u': pl['u'].values, 'v': pl['v'].values,
             'q': pl['q'].values, 'z': pl['z'].values}   # z = geopotential, m2/s2

    lon_raw = d2['longitude'].values.astype(np.float32)
    shift = -int(np.argmin(lon_raw % 360))          # index where 0 deg sits
    lon = np.roll(lon_raw % 360, shift)
    surf  = {k: np.roll(v, shift, axis=-1) for k, v in surf.items()}
    atmos = {k: np.roll(v, shift, axis=-1) for k, v in atmos.items()}

    lat = d2['latitude'].values.astype(np.float32)
    for k, v in surf.items():
        assert v.shape == (721, 1440), f'{tag}/{k} has shape {v.shape}'
    for k, v in atmos.items():
        assert v.shape == (len(LEVELS), 721, 1440), f'{tag}/{k} has shape {v.shape}'
    return surf, atmos, lat, lon

surf_t, atmos_t, LAT, LON = load('t')
surf_p, atmos_p, _,   _   = load('p')

print('grid :', len(LAT), 'x', len(LON))
print('lat  :', LAT[0], '->', LAT[-1], ' (Aurora needs decreasing)')
print('lon  :', LON[0], '->', LON[-1], ' (Aurora needs [0,360) increasing)')
print()
for k, v in surf_t.items():
    print(f'  surf  {k:4s} {str(v.shape):16s} mean {float(v.mean()):10.2f}')
for k, v in atmos_t.items():
    print(f'  atmos {k:4s} {str(v.shape):16s} mean {float(v.mean()):10.2f}')
print()
print('global mean 2m temp:', round(float(surf_t['2t'].mean()) - 273.15, 2), 'degC')

assert LAT[0] > LAT[-1], 'latitude must be decreasing'
assert np.all(np.diff(LON) > 0), 'longitude must be strictly increasing'
assert LON.min() >= 0 and LON.max() < 360, 'longitude must be in [0, 360)'
assert len(LAT) == 721 and len(LON) == 1440, 'expected the global 0.25deg grid'
print('grid checks passed')


## 5. Static variables

`lsm`, `z` (surface geopotential) and `slt` come from Aurora's own static file,
not from the forecast download.


In [ ]:
import pickle
from huggingface_hub import hf_hub_download

static_path = hf_hub_download(repo_id='microsoft/aurora',
                              filename='aurora-0.25-static.pickle')
with open(static_path, 'rb') as f:
    static_raw = pickle.load(f)

print('keys:', list(static_raw.keys()))
static_vars = {k: torch.as_tensor(np.asarray(static_raw[k]), dtype=torch.float32)
               for k in ('lsm', 'z', 'slt')}
for k, v in static_vars.items():
    print(f'  {k:4s} {tuple(v.shape)}')


## 6. Build the global Batch


In [ ]:
from aurora import Batch, Metadata

def t2(a, b):
    """Stack t-6 and t into (batch=1, time=2, ...) float32."""
    return torch.from_numpy(np.stack([a, b])[None].astype(np.float32))

batch = Batch(
    surf_vars={k: t2(surf_p[k], surf_t[k]) for k in SURF},
    static_vars=static_vars,
    atmos_vars={k: t2(atmos_p[k], atmos_t[k]) for k in atmos_t},
    metadata=Metadata(
        lat=torch.from_numpy(LAT),
        lon=torch.from_numpy(LON),
        time=(CYCLE.replace(tzinfo=None),),
        atmos_levels=tuple(LEVELS),
    ),
)

for k, v in batch.surf_vars.items():
    print(f'  surf  {k:4s} {tuple(v.shape)}')
for k, v in batch.atmos_vars.items():
    print(f'  atmos {k:4s} {tuple(v.shape)}')

nbytes = sum(v.numel() * 4 for v in list(batch.surf_vars.values()) + list(batch.atmos_vars.values()))
print(f'batch size: {nbytes / 1024**3:.2f} GiB as float32')


## 7. Load the model

`AuroraV1p5` is upstream's recommended variant. `AuroraPretrained` is the fallback.
`AuroraSmallPretrained` is deliberately **not** an option here.


In [ ]:
import aurora, inspect

available = [c for c in dir(aurora) if c.startswith('Aurora')]
print('available:', available)

# Variant is chosen by INPUT REQUIREMENTS, not just skill:
#   AuroraV1p5        needs 19 surface + 36 static vars -> our batch cannot feed it
#   Aurora            IFS HRES T0 fine-tuned; defaults to exactly our variable set
# Our ICs are ECMWF `oper` step=0 = the HRES analysis at T0, so `Aurora` fits.
PREFER = ('Aurora', 'AuroraPretrained')
MODEL_NAME = next((c for c in PREFER if c in available), None)
if MODEL_NAME is None:
    raise SystemExit(f'No usable Aurora class found in {available}')
assert 'Small' not in MODEL_NAME, 'small model is debug-only, refusing'
cls = getattr(aurora, MODEL_NAME)

# Pure float16 is the main memory lever on a 16 GB card: it halves both the
# weights and the activations. Docs put global 0.25 at ~40 GB in fp32.
#
# Why float16 and NOT bfloat16: Aurora's forward does `batch.type(p.dtype)`,
# which also casts metadata.lon. Verified numerically on the real 1440-point
# grid -- fp16 has ulp 0.25 near 360, exactly the grid spacing, so 359.75 is
# represented EXACTLY. bf16 has ulp 2.0 there, collapsing the top of the grid
# to 358.0 and breaking strict monotonicity, which Aurora's validator rejects.
DTYPE = torch.float16
want = {'use_fp16_safe_attention': True}   # upstream: recommended with fp16
accepted = set(inspect.signature(cls.__init__).parameters)
kwargs = {k: v for k, v in want.items() if k in accepted}
if len(kwargs) < len(want):
    print('note: this build ignores', sorted(set(want) - set(kwargs)))

print('using      :', MODEL_NAME, kwargs, '| dtype', DTYPE)
model = cls(**kwargs)
model.load_checkpoint()
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f'parameters : {params/1e9:.2f} B')
print(f'weights    : {params*2/1024**3:.2f} GiB at fp16'
      f'  (was {params*4/1024**3:.2f} GiB at fp32)')


## 8. The measurement

One step first — autoregressive rollout under `inference_mode` holds roughly constant
memory per step, so step 1 reveals the ceiling. If it survives, we continue to 20
(= 5 days at 6-hourly) and time it.


In [ ]:
import time, gc, os
# Large single allocations fragment the caching allocator; expandable segments
# let it grow a block instead of failing outright.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from aurora import rollout

DEV = 'cuda'
gc.collect(); torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# float16 everywhere. Safe for metadata.lon (see cell 7); bfloat16 is NOT.
model = model.to(DEV).to(DTYPE)
after_weights = torch.cuda.max_memory_allocated() / 1024**3
print(f'after weights to GPU : {after_weights:5.2f} GiB')

batch_gpu = batch.to(DEV).type(DTYPE)
after_batch = torch.cuda.max_memory_allocated() / 1024**3
print(f'after batch to GPU   : {after_batch:5.2f} GiB')

STEPS = 20
preds, timings, peak, failed_at, failure = [], [], after_batch, None, None

t_start = time.time()
try:
    with torch.inference_mode():
        for i, pred in enumerate(rollout(model, batch_gpu, steps=STEPS)):
            timings.append(time.time() - t_start)
            peak = torch.cuda.max_memory_allocated() / 1024**3
            # keep only 2t on CPU; holding 20 global states on GPU would itself OOM
            f32 = pred.surf_vars['2t'][0, -1].float().cpu().numpy()
            if not np.isfinite(f32).all():
                failed_at, failure = i + 1, 'nan'
                print(f'\nNON-FINITE output at step {i+1}: fp16 overflowed.')
                break
            preds.append(f32)
            if i == 0:
                print(f'after step 1         : {peak:5.2f} GiB  ({timings[0]:.1f}s)')
            t_start = time.time()
except torch.cuda.OutOfMemoryError as e:
    failed_at, failure = len(preds) + 1, 'oom'
    # The 'Tried to allocate' figure matters more than the peak: a failed
    # allocation never shows up in max_memory_allocated().
    print(f'\nOOM at step {failed_at}. Full message:')
    print(' ', str(e).split('\n')[0])
    print(f'  reserved {torch.cuda.max_memory_reserved()/1024**3:.2f} GiB,'
          f' allocated {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB')
except Exception as e:
    failed_at, failure = len(preds) + 1, 'error'
    print(f'\nFAILED at step {failed_at} -- this is NOT a memory problem:')
    print(f'  {type(e).__name__}: {e}')
    import traceback; traceback.print_exc()

print()
print('=' * 58)
print(f'  model            : {MODEL_NAME} @ {DTYPE}')
print(f'  steps completed  : {len(preds)} / {STEPS}')
print(f'  PEAK VRAM        : {peak:.2f} GiB')
if timings:
    print(f'  mean s/step      : {sum(timings)/len(timings):.1f}')
    print(f'  est. full run    : {sum(timings)/len(timings)*STEPS/60:.1f} min')
print('=' * 58)


## 9. Physical sanity check

The old synthetic feed claimed an identical **38.6 °C** peak for Hyderabad every day of
the year, in every season. Real output must not look like that.


In [ ]:
if not preds:
    print('No steps completed — nothing to check.')
else:
    def at(field, lat, lon):
        i = int(np.abs(LAT - lat).argmin())
        j = int(np.abs(LON - (lon % 360)).argmin())
        return float(field[i, j]) - 273.15

    SITES = {'Hyderabad': (17.385, 78.4867), 'Warangal': (17.978, 79.594),
             'Khammam': (17.247, 80.151), 'Nizamabad': (18.672, 78.094)}

    print(f'{MODEL_NAME} 2m temperature, init {CYCLE:%Y-%m-%d %H}Z\n')
    hdr = 'lead  ' + '  '.join(f'{s:>10s}' for s in SITES)
    print(hdr); print('-' * len(hdr))
    for i, f in enumerate(preds):
        if i % 2:  # every 12h, keeps it readable
            continue
        row = '  '.join(f'{at(f, *c):10.1f}' for c in SITES.values())
        print(f'+{(i+1)*6:3d}h  {row}')

    hyd = [at(f, *SITES['Hyderabad']) for f in preds]
    lo, hi, spread = min(hyd), max(hyd), max(hyd) - min(hyd)
    print(f'\nHyderabad range: {lo:.1f} to {hi:.1f} degC  (spread {spread:.1f})')

    checks = {
        'plausible for the Deccan (0-50 degC)': -5 < lo and hi < 50,
        'has a real diurnal cycle (spread > 3)': spread > 3,
        'not the synthetic 38.6 degC constant': abs(hi - 38.6) > 0.5 or spread > 3,
        'neighbours differ (not one flat field)':
            abs(at(preds[0], *SITES['Hyderabad']) - at(preds[0], *SITES['Nizamabad'])) > 0.05,
    }
    print()
    for name, ok in checks.items():
        print(f'  [{"PASS" if ok else "FAIL"}] {name}')


## 10. Verdict


In [ ]:
vram_total = sum(torch.cuda.get_device_properties(i).total_memory
                 for i in range(torch.cuda.device_count())) / 1024**3

print(f'Peak {peak:.2f} GiB of {vram_total:.1f} GiB available'
      f' — {len(preds)}/{STEPS} steps\n')

if len(preds) == STEPS:
    print('VERDICT: FEASIBLE on this accelerator.')
    print(f'  Headroom {max(0.0, vram_total - peak):.1f} GiB.')
    print(f'  {sum(timings)/len(timings):.1f} s/step.')
    print('  -> Phase 2: build run_aurora_india.py, slice to Telangana,')
    print('     push to POST /internal/aurora/forecast.')
elif failure == 'nan':
    print('VERDICT: ran, but fp16 overflowed to NaN — numerically unusable.')
    print('  -> Memory is fine; precision is not. Next lever would be fp32')
    print('     activations on a larger GPU, not more Kaggle tuning.')
elif failure == 'error':
    # A crash is not a memory finding. Say so plainly rather than blaming VRAM.
    print('VERDICT: INCONCLUSIVE — the run crashed, it did not run out of memory.')
    print(f'  Peak {peak:.2f} GiB is nowhere near the {vram_total:.1f} GiB limit.')
    print('  -> Fix the error above and re-run. Memory is not the problem.')
elif failure == 'oom':
    print(f'VERDICT: out of memory at step {failed_at}, peak {peak:.2f} GiB.')
    if torch.cuda.device_count() == 1:
        print('  -> Retry with Accelerator -> GPU T4 x2.')
    else:
        print('  -> Would need model sharding across both GPUs. Weigh that against')
        print('     ECMWF AIFS via Open-Meteo (models=ecmwf_aifs025_single):')
        print('     a real operational AI weather model, free, CC-BY-4.0,')
        print('     global, covers all Telangana, no GPU at all.')
else:
    print('VERDICT: no steps ran and no error was recorded — check the cells above.')
